In [10]:
from matplotlib import pyplot as plt
import numpy as np

X_train_final = np.load('X_train_final.npz')['data']
y_train = np.load('y_train.npz')['data']
X_test_final = np.load('X_test_final.npz')['data']
y_test = np.load('y_test.npz')['data']

mean = X_train_final.mean(axis=0)
std = X_train_final.std(axis=0)

X_train = (X_train_final - mean) / std
X_test = (X_test_final - mean) / std


def one_hot(y, num_classes):
    out = np.zeros((len(y), num_classes))
    for i in range(len(y)):
        out[i, int(y[i])] = 1
    return out


def softmax(logits):
    exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    return exps / np.sum(exps, axis=1, keepdims=True)


def cross_entropy_loss(y_onehot, probs):
    eps = 1e-12
    probs = np.clip(probs, eps, 1.0)
    return -np.mean(np.sum(y_onehot * np.log(probs), axis=1))


def accuracy(y_true, probs):
    preds = np.argmax(probs, axis=1)
    return np.mean(preds == y_true)


def gradient_step(X, y_onehot, probs, W, b, lr):
    error = probs - y_onehot
    num_samples = X.shape[0]

    dW = (1 / num_samples) * (X.T @ error)
    db = (1 / num_samples) * np.sum(error, axis=0)

    W -= lr * dW
    b -= lr * db

    return W, b

if __name__ == '__main__':
    y_train_oh = one_hot(y_train, 28)
    y_test_oh = one_hot(y_test, 28)

    n_features = X_train.shape[1]

    W = np.random.randn(n_features, 28)
    b = np.random.randn(28)

    lr = 0.1
    epochs = 5000

    loss_history = []

    for epoch in range(epochs):
        logits = X_train @ W + b
        probs = softmax(logits)

        loss = cross_entropy_loss(y_train_oh, probs)
        loss_history.append(loss)

        W, b = gradient_step(X_train, y_train_oh, probs, W, b, lr)

    plt.plot(loss_history)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss")
    plt.show()

KeyboardInterrupt: 